# 1. Audit the original tobacco dataset
The dataset description in `original_article.pdf`, §2.1 and Table 1, describes eight Mavic Mini campaigns in Mardan, Pakistan, April 2021; plants approximately 15–40 days old; altitude approximately 4 m; reported ground sampling distance 0.1 cm/pixel. Original images are 1920×1080, with non-overlapping 480×352 patches. Semantic labels are background=0, tobacco=1, weed=2. The paper trains on field 147 (864 patches) and tests on the other seven fields (1,656 patches). **We do not implement the paper's detection methodology.**

Our independent experiment uses V2, campaigns 1–5 train, 6 validation, 7–8 test. V1 is checked for duplication, not added as extra training examples. This is distinct from the historical 110-image YOLO split. Semantic masks are not individual-plant annotations.

Enable the audit explicitly. It records byte hashes, decoded-image duplicates, candidate scene overlap, annotation values, and patch-to-source evidence. It does not change inputs. Similarity warnings require recorded inspection before export is allowed.


In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_notebooks').is_dir() and (p / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'dataset_notebooks'))
from runtime import fresh_run, write_json, read_json, sha256, environment, load_model, SAM3_REVISION


In [ ]:
RUN_AUDIT = False
REVIEW_AUDIT_PATH = None  # Or an existing audit.json; inspect/approve without rerunning audit.
DATASET_ROOT = ROOT / 'data/tobacco_original'
RUN_NAME = 'sam3-audit-v1'
EXPECTED = {1:864, 2:936, **{i:120 for i in range(3,9)}}
SPLITS = {i: ('train' if i <= 5 else 'val' if i == 6 else 'test') for i in range(1,9)}


In [ ]:
def audit_dataset(dataset_root, expected, splits):
    import hashlib
    import cv2
    from collections import Counter, defaultdict
    from functools import lru_cache
    @lru_cache(maxsize=16)
    def read_source(path):
        return np.array(Image.open(path).convert('RGB'))
    root = Path(dataset_root)
    v2 = root / 'Tobacco Aerial Dataset V2/Tobacco Aerial Dataset'
    rows, original_masks, source_rows, source_map = [], [], [], {}
    for campaign, count in expected.items():
        c = v2 / f'Campaign no. {campaign}'
        patch = next(p for p in c.iterdir() if p.name.lower() == 'patch images')
        images = {p.stem:p for p in (patch/'data').glob('*.png')}
        masks = {p.stem:p for p in (patch/'mask').glob('*.png')}
        if len(images) != count or images.keys() != masks.keys():
            raise ValueError(f'Campaign {campaign}: missing pairs or unexpected counts')
        originals = sorted((c/'Original 1080P images').glob('*'))
        source_features=[];source_paths=[];source_thumbnails={}
        # Match visual content rather than assuming numeric file ordering.
        for p in originals:
            if p.suffix.lower() not in ('.jpg','.png'): continue
            rgb = np.array(Image.open(p).convert('RGB'))
            thumb = cv2.resize(rgb, (64,36), interpolation=cv2.INTER_AREA)
            source_rows.append({'campaign':campaign,'split':splits[campaign],
                'path':p.relative_to(root).as_posix(), 'sha256':sha256(p),
                'thumbnail':thumb})
            source_map[(campaign,p.stem)] = p
            source_thumbnails[p]=cv2.resize(rgb,(480,270),interpolation=cv2.INTER_AREA)
            for yy in (0,352,704):
                for xx in (0,480,960,1440):
                    feature=cv2.resize(rgb[yy:yy+352,xx:xx+480],(16,12),interpolation=cv2.INTER_AREA)
                    source_features.append(feature.astype('float32').ravel());source_paths.append(p)
        source_features=np.stack(source_features)
        for stem in sorted(images, key=int):
            ip, mp = images[stem], masks[stem]
            rgb = np.array(Image.open(ip).convert('RGB'))
            mask = np.array(Image.open(mp))
            if rgb.shape != (352,480,3) or mask.shape != rgb.shape[:2]:
                raise ValueError(f'Wrong dimensions: {ip}')
            values, counts = np.unique(mask, return_counts=True)
            if not set(values.tolist()) <= {0,1,2}: raise ValueError(f'Unexpected labels: {mp}')
            feature=cv2.resize(rgb,(16,12),interpolation=cv2.INTER_AREA).astype('float32').ravel()
            distances=np.mean((source_features-feature)**2,axis=1)
            candidate_paths=list(dict.fromkeys(source_paths[i] for i in np.argsort(distances)[:6]))
            small_patch=cv2.resize(rgb,(120,88),interpolation=cv2.INTER_AREA)
            matches=[]
            for source in candidate_paths:
                match=cv2.matchTemplate(source_thumbnails[source],small_patch,cv2.TM_SQDIFF_NORMED)
                error,_,coarse_xy,_=cv2.minMaxLoc(match)
                matches.append((error,source,coarse_xy))
            error,source,xy=min(matches,key=lambda v:v[0])
            mapping=None
            if error<.02:
                mapping={'source':source.relative_to(root).as_posix(),'xy_approximate':[xy[0]*4,xy[1]*4],
                    'quarter_scale_normalized_squared_error':error,'method':'appearance_match_not_filename'}
            rows.append({'id':campaign*10000+int(stem),'key':f'c{campaign:02d}_{int(stem):04d}',
                'campaign':campaign,'split':splits[campaign],
                'image':ip.relative_to(root).as_posix(),'semantic_mask':mp.relative_to(root).as_posix(),
                'width':480,'height':352,'image_sha256':sha256(ip),'mask_sha256':sha256(mp),
                'pixel_sha256':hashlib.sha256(rgb.tobytes()).hexdigest(),
                'class_pixels':{str(int(k)):int(v) for k,v in zip(values,counts)},'source_mapping':mapping})
        for folder in ['Labelled 1080P images','Detected vegetation/PixelLabelData']:
            for p in sorted((c/folder).glob('*.png')):
                a=np.array(Image.open(p)); values,counts=np.unique(a,return_counts=True)
                original_masks.append({'path':p.relative_to(root).as_posix(),
                    'shape':list(a.shape),'values':{str(int(k)):int(v) for k,v in zip(values,counts)}})
    by_hash=defaultdict(list)
    for row in rows: by_hash[row['pixel_sha256']].append(row)
    duplicates=[[r['key'] for r in group] for group in by_hash.values() if len(group)>1]
    conflicts=[]
    for group in by_hash.values():
        if len(group)<2:continue
        canonical=min(group,key=lambda r:r['key'])
        for row in group:
            if row is not canonical:row['duplicate_of']=canonical['key']
        if len({r['mask_sha256'] for r in group})>1:conflicts.append([r['key'] for r in group])
    cross_duplicates=[g for g in duplicates if len({r['split'] for r in rows if r['key'] in g})>1]
    # Broad near-duplicate candidate screen. It cannot prove absence of geographic overlap.
    overlap=[]
    for i,a in enumerate(source_rows):
        for b in source_rows[i+1:]:
            if a['split']==b['split']: continue
            error=float(np.mean(np.abs(a['thumbnail'].astype(float)-b['thumbnail'].astype(float))))
            if error < 18:
                overlap.append({'a':a['path'],'b':b['path'],'mean_abs_rgb_error':error})
    v2_hashes={r['image_sha256'] for r in rows}
    v1= root/'Tobacco Aerial Dataset V1'
    v1_files=list(v1.rglob('*.png'))
    return {'schema_version':1,'dataset_root_hint':str(root),'rows':rows,
        'summary':dict(Counter(r['split'] for r in rows)),
        'duplicates':duplicates,'conflicting_semantic_duplicates':conflicts,'cross_split_duplicates':cross_duplicates,
        'overlap_candidates':overlap,'split_review':{'approved':False,'reviewer':'','notes':''},
        'v1_png_count':len(v1_files),'v1_files_matching_v2_images':sum(sha256(p) in v2_hashes for p in v1_files),
        'original_mask_inventory':original_masks,
        'unmapped_patches':[r['key'] for r in rows if r['source_mapping'] is None],
        'sources':[{k:v for k,v in r.items() if k!='thumbnail'} for r in source_rows]}


In [ ]:
audit = None
run = None
if RUN_AUDIT:
    audit = audit_dataset(DATASET_ROOT, EXPECTED, SPLITS)
    run = fresh_run(ROOT, RUN_NAME)
    write_json(run/'audit.json', audit)
    write_json(run/'environment.json', environment())
    print(run, audit['summary'])
    print('Review required:', len(audit['overlap_candidates']), 'overlap candidates;',
          len(audit['cross_split_duplicates']), 'cross-split duplicates;',len(audit['unmapped_patches']),'unmapped patches')
elif REVIEW_AUDIT_PATH is not None:
    audit=read_json(REVIEW_AUDIT_PATH);run=Path(REVIEW_AUDIT_PATH).parent
else:
    print('Audit disabled. No raw data, GPU, downloads, or outputs required.')


## Review the split and label semantics
Run the following preview after the audit. Inspect field context and class overlays (cyan tobacco, yellow weeds). Examine every overlap candidate using its recorded original-image paths. Do not approve known cross-split duplicates. An unresolved patch mapping does not change campaign membership; it prevents claims that patch numbering establishes original-image provenance. The audit does not reconstruct full-resolution ground truth or fill the bottom 24 rows omitted by the 3×352 crop layout.

Inspect conflicting duplicate semantic guides against RGB; review the lowest-key canonical image carefully. Repeated images are excluded from the export. Record reviewer identity, notes about overlap, duplicate conflicts, and the class mapping, and approval below. This is a new artifact; historical split files remain untouched.


In [ ]:
if audit is not None:
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(2,4,figsize=(16,7))
    for campaign,ax in enumerate(axes.flat,1):
        row=next(r for r in audit['rows'] if r['campaign']==campaign)
        im=np.array(Image.open(DATASET_ROOT/row['image']).convert('RGB'))
        m=np.array(Image.open(DATASET_ROOT/row['semantic_mask']))
        overlay=im.copy(); overlay[m==1]=(0,255,255); overlay[m==2]=(255,255,0)
        ax.imshow((.65*im+.35*overlay).astype('uint8'));ax.set_title(f'Campaign {campaign}');ax.axis('off')
    plt.show()


In [ ]:
APPROVE_SPLIT = False
REVIEWER = ''
REVIEW_NOTES = ''
if APPROVE_SPLIT:
    if audit is None or not REVIEWER.strip() or not REVIEW_NOTES.strip():
        raise ValueError('Run audit and record reviewer plus overlap/class-mapping inspection notes.')
    if audit['cross_split_duplicates']: raise ValueError('Resolve cross-split duplicates before proceeding.')
    audit['split_review']={'approved':True,'reviewer':REVIEWER,'notes':REVIEW_NOTES}
    write_json(run/'audit.json',audit)
